# Bracket matching — 00: behavioral evaluation

Second stage of the SantaCoder mechanistic-interpretability investigation. Before
looking for circuits we must establish that the model *has* the behavior and define a
clean evaluation task, mirroring the induction-head workflow in
`scripts/benchmark.py`.

**Task.** Each prompt is a minimal-pair code snippet with exactly **one** pending opener,
so the correct next token is a specific closer (`)`, `]`, or `}`). We score the model with
the *closer logit difference*

$$\Delta = \text{logit}(\text{correct closer}) - \max_{c \neq \text{correct}} \text{logit}(c)$$

A positive $\Delta$ means the model favours the right closer over the two wrong ones.

The prompt list defined below is the **shared dataset**: `01_analysis.ipynb` copies it
verbatim so the two notebooks stay independent but aligned.

## Load SantaCoder (CPU)

We apply the same local-cache compatibility patch the induction-head experiments use
(`scripts/patch_santacoder.py`) before loading via TransformerLens.

In [1]:
import sys
from pathlib import Path

import torch
from transformer_lens import HookedTransformer
from transformer_lens import utils as tl_utils

# Locate the repository root so we can import the SantaCoder compat patch regardless of
# whether Jupyter was started from the repo root or from this notebook's directory.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.patch_santacoder import ensure_santacoder_transformers_compat

torch.set_grad_enabled(False)

MODEL_NAME = "santacoder"
DEVICE = "cpu"

# Patch the local HF cache (never the published files) before loading the model.
for line in ensure_santacoder_transformers_compat():
    print(line)

model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()
print(f"loaded {MODEL_NAME}: {model.cfg.n_layers} layers x {model.cfg.n_heads} heads")

already compatible: /home/estevao/.cache/huggingface/hub/models--bigcode--santacoder/snapshots/bb3be599767d93ce716293e9193c027e855a9524/modeling_gpt2_mq.py
synced snapshot to dynamic module cache: /home/estevao/.cache/huggingface/modules/transformers_modules/bigcode/santacoder/bb3be599767d93ce716293e9193c027e855a9524/modeling_gpt2_mq.py
already compatible: /home/estevao/.cache/huggingface/modules/transformers_modules/bigcode/santacoder/bb3be599767d93ce716293e9193c027e855a9524/modeling_gpt2_mq.py


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model santacoder into HookedTransformer
loaded santacoder: 24 layers x 16 heads


## Shared prompt dataset

15 prompts, 5 per closer type, varying nesting depth and surrounding context across
Python and JavaScript styles. Snippets deliberately contain no brackets inside string
literals, so a simple stack scan can later (in `01`) locate the pending opener
unambiguously. Each `code` string ends exactly where the closer should be predicted.

In [2]:
# === SHARED PROMPT DATASET (reused verbatim by 01_analysis.ipynb) ===
BRACKET_PROMPTS = [
    # --- round parens ) ---
    {"code": "def foo(x, y", "closer": ")"},
    {"code": "result = max(first, second", "closer": ")"},
    {"code": "total = sum(values", "closer": ")"},
    {"code": "if (x > 0", "closer": ")"},
    {"code": "print(count, total", "closer": ")"},
    # --- square brackets ] ---
    {"code": "first = items[0", "closer": "]"},
    {"code": "cell = matrix[i][j", "closer": "]"},
    {"code": "data = [1, 2, 3", "closer": "]"},
    {"code": "last = arr[len(arr) - 1", "closer": "]"},
    {"code": "value = lookup[keys[0]", "closer": "]"},
    # --- curly braces } ---
    {"code": "if (x > 0) {", "closer": "}"},
    {"code": "const obj = {", "closer": "}"},
    {"code": "function add(a, b) {", "closer": "}"},
    {"code": "payload = {'key': value", "closer": "}"},
    {"code": "while (i < n) {", "closer": "}"},
]
print(f"{len(BRACKET_PROMPTS)} prompts")

15 prompts


## Closer token ids and tokenization sanity check

Each closer must be a single SantaCoder token so the logit lookup is exact.

In [3]:
CLOSERS = [")", "]", "}"]

def single_token_id(text):
    ids = model.to_tokens(text, prepend_bos=False)[0].tolist()
    assert len(ids) == 1, f"{text!r} is not a single token: {ids}"
    return ids[0]

closer_ids = {c: single_token_id(c) for c in CLOSERS}
print("closer token ids:", closer_ids)

for p in BRACKET_PROMPTS:
    n = model.to_tokens(p["code"]).shape[1]
    assert n >= 2, p
print("all prompts tokenize cleanly")

closer token ids: {')': 8, ']': 60, '}': 92}
all prompts tokenize cleanly


## Logit-difference evaluation

In [4]:
def closer_logit_diff(code, correct):
    tokens = model.to_tokens(code)
    logits = model(tokens)
    final = logits[0, -1]
    correct_logit = final[closer_ids[correct]].item()
    best_other = max(final[closer_ids[c]].item() for c in CLOSERS if c != correct)
    return correct_logit - best_other

records = []
for p in BRACKET_PROMPTS:
    diff = closer_logit_diff(p["code"], p["closer"])
    records.append({"code": p["code"], "closer": p["closer"], "logit_diff": diff})
    flag = "  <-- WRONG" if diff < 0 else ""
    print(f"{diff:+.3f}  [{p['closer']}]  {p['code']}{flag}")

+8.254  [)]  def foo(x, y


+10.429  [)]  result = max(first, second


+13.887  [)]  total = sum(values


+10.586  [)]  if (x > 0


+10.445  [)]  print(count, total


...

+3.391  [}]  if (x > 0) {


+3.419  [}]  const obj = {


+2.194  [}]  function add(a, b) {


+7.323  [}]  payload = {'key': value


+0.768  [}]  while (i < n) {


In [5]:
import statistics

mean_diff = statistics.fmean(r["logit_diff"] for r in records)
per_type = {
    c: statistics.fmean([r["logit_diff"] for r in records if r["closer"] == c])
    for c in CLOSERS
}
failures = [r for r in records if r["logit_diff"] < 0]

print(f"mean logit diff across {len(records)} prompts: {mean_diff:+.3f}\n")
print("per-closer-type mean:")
for c in CLOSERS:
    n = sum(1 for r in records if r["closer"] == c)
    print(f"  {c} : {per_type[c]:+.3f}  (n={n})")
print(f"\nfailures (logit_diff < 0): {len(failures)}")
for r in failures:
    print(f"  {r['logit_diff']:+.3f}  [{r['closer']}]  {r['code']}")

mean logit diff across 15 prompts: +8.534

per-closer-type mean:
  ) : +10.720  (n=5)
  ] : +11.463  (n=5)
  } : +3.419  (n=5)

failures (logit_diff < 0): 0


In [6]:
import json

out_dir = ROOT / "outputs" / "bracket_matching"
out_dir.mkdir(parents=True, exist_ok=True)
payload = {
    "model_name": MODEL_NAME,
    "n_prompts": len(records),
    "mean_logit_diff": mean_diff,
    "per_closer_type_mean": per_type,
    "n_failures": len(failures),
    "failures": failures,
    "prompts": records,
    "closer_token_ids": closer_ids,
}
out_path = out_dir / "behavioral.json"
out_path.write_text(json.dumps(payload, indent=2))
print(f"saved {out_path}")

saved /home/estevao/mechinterp/outputs/bracket_matching/behavioral.json


## Interpretation

Read the printed numbers above against these questions:

- **Does SantaCoder show reliable bracket matching?** A clearly positive *mean logit
  difference* with few or no failures means the model robustly prefers the correct closer
  over the two distractors at the prediction site. This is the behavioral pre-condition
  for the circuit search in `01`; if the mean were near zero we would not expect to find a
  clean causal mechanism.
- **Per-type breakdown.** Compare the `)`, `]`, `}` means. `)` typically benefits from the
  most training support (function calls / defs dominate code), so a weaker `}` column would
  be unsurprising — JavaScript-style braces are rarer in SantaCoder's distribution and the
  immediate post-`{` token is often a newline rather than a closer.
- **Failure pattern.** Inspect any prompts with negative $\Delta$. The interesting question
  is whether failures cluster by *closer type* (a representational gap) or by *nesting
  depth* (the model loses track of which opener is pending). Deeply nested or
  JS-brace prompts are the most likely failure sites and motivate the head-level analysis
  in `01_analysis.ipynb`.

> Per the project's standing caveat, these are research inputs: the saved
> `outputs/bracket_matching/behavioral.json` is the evidence record, not a finished claim.